# hftbacktest - TW ETF

Thin runner for Taiwan TW ETF top-5 experiments.

## Setup

Set the symbol and time range, convert L2/top-5 data to hftbacktest events, then build shared backtest config.

In [5]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts import tw_stock_data_to_npz as tw_npz
tw_npz = importlib.reload(tw_npz)
convert_tw_etf_to_npz = tw_npz.convert_tw_etf_to_npz
default_daily_parquet_dir = tw_npz.default_daily_parquet_dir
from scripts.tw_stock_hftbacktest import BacktestConfig, import_hftbacktest
from scripts.tw_stock_strategies import (
    DEFAULT_QUEUE_MODELS,
    run_aggressive_fill_strategy,
    run_queue_model_comparison,
    run_level_queue_model_comparison,
)


In [6]:
SYMBOL = "0050"
START_DATE = "2026-06-30"
END_DATE = START_DATE
START_TIME = "09:30:00"
END_TIME = "10:00:00"
SOURCE_KIND = "etf"
DATA_DIR = default_daily_parquet_dir(ROOT, SOURCE_KIND)
TICK_SIZE = 0.05
CONTRACT_SIZE = 1000.0
PRICE_ONLY_DEPTH_QTY = 1.0

DATA_FILE, event_data = convert_tw_etf_to_npz(
    symbol=SYMBOL,
    start_date=START_DATE,
    end_date=END_DATE,
    start_time=START_TIME,
    end_time=END_TIME,
    workspace_root=ROOT,
    daily_parquet_dir=DATA_DIR,
    price_only_depth_qty=PRICE_ONLY_DEPTH_QTY,
)

hbtpkg = import_hftbacktest(ROOT)
CONFIG = BacktestConfig(
    data=DATA_FILE,
    order_latency_ns=0,
    tick_size=TICK_SIZE,
    contract_size=CONTRACT_SIZE,
)
QUEUE_MODELS = DEFAULT_QUEUE_MODELS
DATA_FILE, DATA_DIR, TICK_SIZE, CONTRACT_SIZE


input_rows=13163
converted_rows=13163
skipped_symbol_rows=0
skipped_status_rows=0
skipped_time_rows=0
raw_events=160079
output_events=160079
depth_events=157824
trade_events=2255
opening_jump_qty=0.0
first_exch_ts=1782783000001941000
last_exch_ts=1782784799923799000
min_feed_latency=0
max_feed_latency=0
qa_rows_checked=1000
best_bid_mismatches=0
best_ask_mismatches=0
trade_qty_mismatches=0
output=C:\Users\zoufuc\Desktop\hftbacktest\data\tw_etf_events\0050_20260630_093000_100000.npz


(WindowsPath('C:/Users/zoufuc/Desktop/hftbacktest/data/tw_etf_events/0050_20260630_093000_100000.npz'),
 WindowsPath('//DC_TW/taiwan_stock/ticks_parquet_etf'),
 0.05,
 1000.0)

## Strategy 1: Aggressive Fill at BBO

Buy at best ask, then sell at best bid. This should fill immediately by design.

In [7]:
strategy1_output = run_aggressive_fill_strategy(
    CONFIG,
    hbtpkg,
    event_data,
    qty=1.0,
    round_trips=1,
)
strategy1_summary = strategy1_output[
    [
        "label", "side", "order_id", "price", "exec_price", "exec_qty",
        "send_order_time", "fill_time", "position", "balance", "equity",
        "num_trades", "trading_volume", "trading_value",
    ]
]
strategy1_summary


,label,side,order_id,price,exec_price,exec_qty,send_order_time,fill_time,position,balance,equity,num_trades,trading_volume,trading_value
0,initial_bbo,NaN,NaN,NaN,<NA>,<NA>,NaT,NaT,0.0,0.0,0.0,0,0.0,0.0
1,before_buy,buy,10001.0,107.80,<NA>,<NA>,NaT,NaT,0.0,0.0,0.0,0,0.0,0.0
2,after_buy,buy,10001.0,107.80,107.8,1.0,2026-06-30 09:30:01.001941+08:00,2026-06-30 09:30:01.001941+08:00,1.0,-107800.0,-25.0,1,1.0,107800.0
3,before_sell,sell,10002.0,107.75,<NA>,<NA>,NaT,NaT,1.0,-107800.0,-25.0,1,1.0,107800.0
4,after_sell,sell,10002.0,107.75,107.75,1.0,2026-06-30 09:30:01.001941+08:00,2026-06-30 09:30:01.001941+08:00,0.0,-50.0,-50.0,2,2.0,215550.0
5,final_state,NaN,NaN,NaN,<NA>,<NA>,NaT,NaT,0.0,-50.0,-50.0,2,2.0,215550.0


## Strategy 2: Passive Bid1/Ask1 Queue Model Comparison

Submit passive buy at bid1 and passive sell at ask1. Compare fill timestamps across queue models.

In [8]:
strategy2_output, strategy2_fill_comparison = run_queue_model_comparison(
    CONFIG,
    hbtpkg,
    event_data,
    queue_models=QUEUE_MODELS,
    qty=1.0,
)
strategy2_summary = strategy2_fill_comparison[
    [
        "queue_model", "side", "order_id", "price", "exec_price", "exec_qty",
        "send_order_time", "fill_time", "time_to_fill_s", "queue_model_fill_delta_ns",
        "position", "balance", "equity",
    ]
]
strategy2_summary


,queue_model,side,order_id,price,exec_price,exec_qty,send_order_time,fill_time,time_to_fill_s,queue_model_fill_delta_ns,position,balance,equity
0,log_prob,buy,20001.0,107.75,107.75,1.0,2026-06-30 09:30:01.001941+08:00,2026-06-30 09:30:05.476835+08:00,4.474894,0,0.0,50.0,50.0
1,risk_adverse,buy,20001.0,107.75,107.75,1.0,2026-06-30 09:30:01.001941+08:00,2026-06-30 09:30:05.476835+08:00,4.474894,0,0.0,50.0,50.0
2,log_prob,sell,20002.0,107.80,107.8,1.0,2026-06-30 09:30:01.001941+08:00,2026-06-30 09:30:03.101313+08:00,2.099372,0,-1.0,107800.0,25.0
3,risk_adverse,sell,20002.0,107.80,107.8,1.0,2026-06-30 09:30:01.001941+08:00,2026-06-30 09:30:03.101313+08:00,2.099372,0,-1.0,107800.0,25.0


## Strategy 3: Passive Sell5

Submit one passive order at a fixed book level with a longer observation window.

In [10]:
strategy3_output, strategy3_comparison = run_level_queue_model_comparison(
    CONFIG,
    hbtpkg,
    event_data,
    queue_models=QUEUE_MODELS,
    side="sell",
    level=5,
    qty=3.0,
    max_window_s=6 * 60 * 60,
)
strategy3_summary = strategy3_comparison[
    [
        "queue_model", "side", "level", "actual_level", "price", "qty",
        "send_best_bid", "send_best_ask", "send_order_time", "fill_time", "was_filled",
        "time_to_fill_s", "queue_model_fill_delta_ns", "fill_step", "exec_price", "exec_qty",
        "position", "balance", "equity",
    ]
]
strategy3_summary


,queue_model,side,level,actual_level,price,qty,send_best_bid,send_best_ask,send_order_time,fill_time,was_filled,time_to_fill_s,queue_model_fill_delta_ns,fill_step,exec_price,exec_qty,position,balance,equity
0,log_prob,sell,5,5,108.0,3.0,107.75,107.8,2026-06-30 09:30:01.001941+08:00,2026-06-30 09:32:46.798415+08:00,True,165.796474,0,166,108.0,3.0,-3.0,324000.0,75.0
1,risk_adverse,sell,5,5,108.0,3.0,107.75,107.8,2026-06-30 09:30:01.001941+08:00,2026-06-30 09:32:46.798415+08:00,True,165.796474,0,166,108.0,3.0,-3.0,324000.0,75.0
